# District Heating MILP Optimizer — Prototype

This notebook demonstrates the Mixed-Integer Linear Programming (MILP) approach for optimal dispatch of a district heating system.

**Problem:** Minimize the total operational cost of meeting thermal demand over a 24-hour horizon by dispatching:
- **Heat Pump** (HP) — consumes electricity, produces heat via COP
- **Condensing Boiler** — burns gas
- **Combined Heat & Power** (CHP) — burns gas, produces heat + electricity (sold to grid)
- **Thermal Storage** — 200 MWh buffer, shifts load across time

**Why MILP?** The on/off decisions (binary), minimum up/downtime constraints, and startup costs make this a mixed-integer problem.

See `docs/optimization/optimization_problem.md` for the full mathematical formulation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pyomo.environ as pyo

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ─── Asset Parameters ───────────────────────────────────────────────

PARAMS = {
    # Time
    "T": 96,               # intervals (24h at 15-min resolution)
    "dt": 0.25,            # hours per interval

    # Gas grid
    "gas_price": 35.0,     # €/MWh_Hs
    "co2_factor": 0.201,   # tCO2/MWh_Hs
    "co2_price": 60.0,     # €/tCO2

    # Thermal storage
    "sto_capacity": 200.0,     # MWh_th
    "sto_charge_max": 15.0,    # MW_th
    "sto_discharge_max": 15.0, # MW_th
    "sto_loss": 0.000125,      # MWh_th per 15-min interval
    "sto_soc_init": 200.0,     # MWh_th (used only for the very first day)

    # Heat pump
    "hp_p_el_max": 8.0,    # MW_el
    "hp_p_el_min": 1.0,    # MW_el
    "hp_cop": 3.5,         # MW_th / MW_el

    # Condensing boiler
    "boiler_q_max": 12.0,  # MW_th
    "boiler_q_min": 2.0,   # MW_th
    "boiler_eff": 0.97,    # MW_th / MW_Hs
    "boiler_min_up": 4,    # intervals (= 1 hour)
    "boiler_min_down": 4,  # intervals (= 1 hour)

    # CHP
    "chp_p_el_max": 6.0,   # MW_el
    "chp_p_el_min": 2.0,   # MW_el
    "chp_eff_el": 0.4,     # MW_el / MW_Hs
    "chp_eff_th": 0.48,    # MW_th / MW_Hs
    "chp_min_up": 8,       # intervals (= 2 hours)
    "chp_min_down": 8,     # intervals (= 2 hours)
    "chp_startup_cost": 600.0,  # € per startup
}

# Derived
PARAMS["gas_cost"] = PARAMS["gas_price"] + PARAMS["co2_factor"] * PARAMS["co2_price"]  # 47.06 €/MWh_Hs
PARAMS["chp_heat_power_ratio"] = PARAMS["chp_eff_th"] / PARAMS["chp_eff_el"]  # 1.2


def build_model(demand_mw: list[float], price_eur: list[float],
                initial_states: dict | None = None) -> pyo.ConcreteModel:
    """
    Build the Pyomo MILP model.

    Parameters
    ----------
    demand_mw : list of 24 floats
        Hourly heat demand in MW_th.
    price_eur : list of 24 floats
        Hourly DA electricity price in €/MWh.
    initial_states : dict, optional
        Rolling horizon carry-over. Keys: z_hp_0, z_boiler_0, z_chp_0,
        boiler_time_in_state_0, chp_time_in_state_0, soc_0.
        If None, all units start OFF and storage at sto_soc_init.
    """
    p = PARAMS
    T = p["T"]
    dt = p["dt"]

    if len(demand_mw) != 24 or len(price_eur) != 24:
        raise ValueError("Need exactly 24 hourly values for demand and price")

    demand_15min = np.repeat(demand_mw, 4)
    price_15min = np.repeat(price_eur, 4)

    if initial_states is None:
        initial_states = {
            "z_hp_0": 0, "z_boiler_0": 0, "z_chp_0": 0,
            "boiler_time_in_state_0": 999,
            "chp_time_in_state_0": 999,
            "soc_0": p["sto_soc_init"],
        }

    soc_init = initial_states.get("soc_0", p["sto_soc_init"])

    m = pyo.ConcreteModel("DistrictHeatingMILP")
    m.T_set = pyo.RangeSet(1, T)

    m.demand = pyo.Param(m.T_set, initialize={t: demand_15min[t - 1] for t in range(1, T + 1)})
    m.da_price = pyo.Param(m.T_set, initialize={t: price_15min[t - 1] for t in range(1, T + 1)})

    # Decision Variables
    m.z_hp = pyo.Var(m.T_set, within=pyo.Binary)
    m.P_hp_el = pyo.Var(m.T_set, within=pyo.NonNegativeReals, bounds=(0, p["hp_p_el_max"]))
    m.Q_hp = pyo.Var(m.T_set, within=pyo.NonNegativeReals)

    m.z_boiler = pyo.Var(m.T_set, within=pyo.Binary)
    m.s_boiler = pyo.Var(m.T_set, within=pyo.Binary)
    m.d_boiler = pyo.Var(m.T_set, within=pyo.Binary)
    m.Q_boiler = pyo.Var(m.T_set, within=pyo.NonNegativeReals, bounds=(0, p["boiler_q_max"]))
    m.F_boiler = pyo.Var(m.T_set, within=pyo.NonNegativeReals)

    m.z_chp = pyo.Var(m.T_set, within=pyo.Binary)
    m.s_chp = pyo.Var(m.T_set, within=pyo.Binary)
    m.d_chp = pyo.Var(m.T_set, within=pyo.Binary)
    m.P_chp_el = pyo.Var(m.T_set, within=pyo.NonNegativeReals, bounds=(0, p["chp_p_el_max"]))
    m.Q_chp = pyo.Var(m.T_set, within=pyo.NonNegativeReals)
    m.F_chp = pyo.Var(m.T_set, within=pyo.NonNegativeReals)

    m.y_sto = pyo.Var(m.T_set, within=pyo.Binary)
    m.Q_charge = pyo.Var(m.T_set, within=pyo.NonNegativeReals, bounds=(0, p["sto_charge_max"]))
    m.Q_discharge = pyo.Var(m.T_set, within=pyo.NonNegativeReals, bounds=(0, p["sto_discharge_max"]))
    m.SoC = pyo.Var(pyo.RangeSet(0, T), within=pyo.NonNegativeReals, bounds=(0, p["sto_capacity"]))

    # Objective
    def objective_rule(m):
        return sum(
            m.P_hp_el[t] * m.da_price[t] * dt
            + m.F_boiler[t] * p["gas_cost"] * dt
            + m.F_chp[t] * p["gas_cost"] * dt
            + p["chp_startup_cost"] * m.s_chp[t]
            - m.P_chp_el[t] * m.da_price[t] * dt
            for t in m.T_set
        )
    m.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

    # Energy Balance
    m.energy_balance = pyo.Constraint(m.T_set, rule=lambda m, t:
        m.Q_hp[t] + m.Q_boiler[t] + m.Q_chp[t]
        + m.Q_discharge[t] - m.Q_charge[t] == m.demand[t])

    # Storage — initial SoC from carry-over, no terminal constraint
    m.soc_init = pyo.Constraint(expr=m.SoC[0] == soc_init)

    m.charge_exclusive = pyo.Constraint(m.T_set, rule=lambda m, t:
        m.Q_charge[t] <= p["sto_charge_max"] * m.y_sto[t])
    m.discharge_exclusive = pyo.Constraint(m.T_set, rule=lambda m, t:
        m.Q_discharge[t] <= p["sto_discharge_max"] * (1 - m.y_sto[t]))

    m.soc_dynamics = pyo.Constraint(m.T_set, rule=lambda m, t:
        m.SoC[t] == m.SoC[t - 1] + m.Q_charge[t] * dt - m.Q_discharge[t] * dt - p["sto_loss"])

    # Heat Pump
    m.hp_min = pyo.Constraint(m.T_set, rule=lambda m, t: m.P_hp_el[t] >= p["hp_p_el_min"] * m.z_hp[t])
    m.hp_max = pyo.Constraint(m.T_set, rule=lambda m, t: m.P_hp_el[t] <= p["hp_p_el_max"] * m.z_hp[t])
    m.hp_thermal = pyo.Constraint(m.T_set, rule=lambda m, t: m.Q_hp[t] == p["hp_cop"] * m.P_hp_el[t])

    # Boiler
    m.boiler_min = pyo.Constraint(m.T_set, rule=lambda m, t: m.Q_boiler[t] >= p["boiler_q_min"] * m.z_boiler[t])
    m.boiler_max = pyo.Constraint(m.T_set, rule=lambda m, t: m.Q_boiler[t] <= p["boiler_q_max"] * m.z_boiler[t])
    m.boiler_fuel = pyo.Constraint(m.T_set, rule=lambda m, t: m.F_boiler[t] == m.Q_boiler[t] / p["boiler_eff"])

    z_boiler_0 = initial_states["z_boiler_0"]
    boiler_tis = initial_states.get("boiler_time_in_state_0", 999)
    if z_boiler_0 == 1 and boiler_tis < p["boiler_min_up"]:
        remaining = p["boiler_min_up"] - boiler_tis
        for t_fix in range(1, min(remaining + 1, T + 1)):
            m.z_boiler[t_fix].fix(1)
    elif z_boiler_0 == 0 and boiler_tis < p["boiler_min_down"]:
        remaining = p["boiler_min_down"] - boiler_tis
        for t_fix in range(1, min(remaining + 1, T + 1)):
            m.z_boiler[t_fix].fix(0)

    def boiler_startup_rule(m, t):
        z_prev = z_boiler_0 if t == 1 else m.z_boiler[t - 1]
        return m.s_boiler[t] >= m.z_boiler[t] - z_prev
    def boiler_shutdown_rule(m, t):
        z_prev = z_boiler_0 if t == 1 else m.z_boiler[t - 1]
        return m.d_boiler[t] >= z_prev - m.z_boiler[t]
    m.boiler_startup = pyo.Constraint(m.T_set, rule=boiler_startup_rule)
    m.boiler_shutdown = pyo.Constraint(m.T_set, rule=boiler_shutdown_rule)

    L_up_b = p["boiler_min_up"]
    m.boiler_min_uptime = pyo.Constraint(m.T_set, rule=lambda m, t:
        sum(m.s_boiler[i] for i in range(max(1, t - L_up_b + 1), t + 1)) <= m.z_boiler[t])
    L_dn_b = p["boiler_min_down"]
    m.boiler_min_downtime = pyo.Constraint(m.T_set, rule=lambda m, t:
        sum(m.d_boiler[i] for i in range(max(1, t - L_dn_b + 1), t + 1)) <= 1 - m.z_boiler[t])

    # CHP
    m.chp_min = pyo.Constraint(m.T_set, rule=lambda m, t: m.P_chp_el[t] >= p["chp_p_el_min"] * m.z_chp[t])
    m.chp_max = pyo.Constraint(m.T_set, rule=lambda m, t: m.P_chp_el[t] <= p["chp_p_el_max"] * m.z_chp[t])
    m.chp_thermal = pyo.Constraint(m.T_set, rule=lambda m, t:
        m.Q_chp[t] == p["chp_heat_power_ratio"] * m.P_chp_el[t])
    m.chp_fuel = pyo.Constraint(m.T_set, rule=lambda m, t: m.F_chp[t] == m.P_chp_el[t] / p["chp_eff_el"])

    z_chp_0 = initial_states["z_chp_0"]
    chp_tis = initial_states.get("chp_time_in_state_0", 999)
    if z_chp_0 == 1 and chp_tis < p["chp_min_up"]:
        remaining = p["chp_min_up"] - chp_tis
        for t_fix in range(1, min(remaining + 1, T + 1)):
            m.z_chp[t_fix].fix(1)
    elif z_chp_0 == 0 and chp_tis < p["chp_min_down"]:
        remaining = p["chp_min_down"] - chp_tis
        for t_fix in range(1, min(remaining + 1, T + 1)):
            m.z_chp[t_fix].fix(0)

    def chp_startup_rule(m, t):
        z_prev = z_chp_0 if t == 1 else m.z_chp[t - 1]
        return m.s_chp[t] >= m.z_chp[t] - z_prev
    def chp_shutdown_rule(m, t):
        z_prev = z_chp_0 if t == 1 else m.z_chp[t - 1]
        return m.d_chp[t] >= z_prev - m.z_chp[t]
    m.chp_startup = pyo.Constraint(m.T_set, rule=chp_startup_rule)
    m.chp_shutdown = pyo.Constraint(m.T_set, rule=chp_shutdown_rule)

    L_up_c = p["chp_min_up"]
    m.chp_min_uptime = pyo.Constraint(m.T_set, rule=lambda m, t:
        sum(m.s_chp[i] for i in range(max(1, t - L_up_c + 1), t + 1)) <= m.z_chp[t])
    L_dn_c = p["chp_min_down"]
    m.chp_min_downtime = pyo.Constraint(m.T_set, rule=lambda m, t:
        sum(m.d_chp[i] for i in range(max(1, t - L_dn_c + 1), t + 1)) <= 1 - m.z_chp[t])

    return m


def solve(model: pyo.ConcreteModel, time_limit: int = 120) -> dict:
    """Solve the model and return results including final_states for rolling horizon."""
    solver = pyo.SolverFactory("appsi_highs")
    solver.options["time_limit"] = time_limit

    result = solver.solve(model, tee=False)

    status = result.solver.termination_condition
    if status not in (pyo.TerminationCondition.optimal, pyo.TerminationCondition.feasible):
        return {"status": str(status), "feasible": False}

    T = PARAMS["T"]
    dt = PARAMS["dt"]

    hours = [(t - 1) * dt for t in range(1, T + 1)]
    results = {
        "status": str(status), "feasible": True,
        "objective_eur": pyo.value(model.obj), "hours": hours,
        "demand": [pyo.value(model.demand[t]) for t in model.T_set],
        "da_price": [pyo.value(model.da_price[t]) for t in model.T_set],
        "hp_q_th": [pyo.value(model.Q_hp[t]) for t in model.T_set],
        "hp_p_el": [pyo.value(model.P_hp_el[t]) for t in model.T_set],
        "hp_on": [int(round(pyo.value(model.z_hp[t]))) for t in model.T_set],
        "boiler_q_th": [pyo.value(model.Q_boiler[t]) for t in model.T_set],
        "boiler_on": [int(round(pyo.value(model.z_boiler[t]))) for t in model.T_set],
        "chp_q_th": [pyo.value(model.Q_chp[t]) for t in model.T_set],
        "chp_p_el": [pyo.value(model.P_chp_el[t]) for t in model.T_set],
        "chp_on": [int(round(pyo.value(model.z_chp[t]))) for t in model.T_set],
        "chp_startups": sum(int(round(pyo.value(model.s_chp[t]))) for t in model.T_set),
        "sto_charge": [pyo.value(model.Q_charge[t]) for t in model.T_set],
        "sto_discharge": [pyo.value(model.Q_discharge[t]) for t in model.T_set],
        "soc": [pyo.value(model.SoC[t]) for t in range(T + 1)],
    }

    # Final states for rolling horizon carry-over
    z_boiler_final = int(round(pyo.value(model.z_boiler[T])))
    boiler_tis = 0
    for t_back in range(T, 0, -1):
        if int(round(pyo.value(model.z_boiler[t_back]))) == z_boiler_final:
            boiler_tis += 1
        else:
            break

    z_chp_final = int(round(pyo.value(model.z_chp[T])))
    chp_tis = 0
    for t_back in range(T, 0, -1):
        if int(round(pyo.value(model.z_chp[t_back]))) == z_chp_final:
            chp_tis += 1
        else:
            break

    results["final_states"] = {
        "z_hp_0": int(round(pyo.value(model.z_hp[T]))),
        "z_boiler_0": z_boiler_final,
        "z_chp_0": z_chp_final,
        "boiler_time_in_state_0": boiler_tis,
        "chp_time_in_state_0": chp_tis,
        "soc_0": pyo.value(model.SoC[T]),
    }

    gas_cost = PARAMS["gas_cost"]
    results["cost_hp"] = sum(results["hp_p_el"][i] * results["da_price"][i] * dt for i in range(T))
    results["cost_boiler"] = sum(results["boiler_q_th"][i] / PARAMS["boiler_eff"] * gas_cost * dt for i in range(T))
    results["cost_chp_fuel"] = sum(results["chp_p_el"][i] / PARAMS["chp_eff_el"] * gas_cost * dt for i in range(T))
    results["cost_chp_startup"] = results["chp_startups"] * PARAMS["chp_startup_cost"]
    results["revenue_chp"] = sum(results["chp_p_el"][i] * results["da_price"][i] * dt for i in range(T))
    results["cost_total"] = (results["cost_hp"] + results["cost_boiler"]
                             + results["cost_chp_fuel"] + results["cost_chp_startup"]
                             - results["revenue_chp"])
    return results


def print_summary(results: dict):
    """Print a human-readable summary of the optimization results."""
    if not results["feasible"]:
        print(f"Solver status: {results['status']} — no feasible solution found.")
        return

    print(f"Solver status: {results['status']}")
    print(f"Total cost: {results['objective_eur']:,.2f} €")
    print()
    print("Cost breakdown:")
    print(f"  Heat pump electricity : {results['cost_hp']:>10,.2f} €")
    print(f"  Boiler fuel + CO2     : {results['cost_boiler']:>10,.2f} €")
    print(f"  CHP fuel + CO2        : {results['cost_chp_fuel']:>10,.2f} €")
    print(f"  CHP startup costs     : {results['cost_chp_startup']:>10,.2f} €")
    print(f"  CHP electricity rev.  : {results['revenue_chp']:>10,.2f} € (revenue)")
    print(f"  ─────────────────────────────────────")
    print(f"  Net total             : {results['cost_total']:>10,.2f} €")
    print()

    T = PARAMS["T"]
    dt = PARAMS["dt"]
    hp_energy = sum(results["hp_q_th"]) * dt
    boiler_energy = sum(results["boiler_q_th"]) * dt
    chp_energy = sum(results["chp_q_th"]) * dt
    demand_energy = sum(results["demand"]) * dt
    charge_energy = sum(results["sto_charge"]) * dt
    discharge_energy = sum(results["sto_discharge"]) * dt

    print("Thermal energy delivered (MWh_th):")
    print(f"  Heat pump    : {hp_energy:>8.1f}")
    print(f"  Boiler       : {boiler_energy:>8.1f}")
    print(f"  CHP          : {chp_energy:>8.1f}")
    print(f"  Sto charge   : {charge_energy:>8.1f}")
    print(f"  Sto discharge: {discharge_energy:>8.1f}")
    print(f"  Demand total : {demand_energy:>8.1f}")
    print()

    hp_hours = sum(results["hp_on"]) * dt
    boiler_hours = sum(results["boiler_on"]) * dt
    chp_hours = sum(results["chp_on"]) * dt
    print(f"Operating hours: HP={hp_hours:.1f}h, Boiler={boiler_hours:.1f}h, CHP={chp_hours:.1f}h")
    print(f"CHP startups: {results['chp_startups']}")
    print(f"Storage SoC: start={results['soc'][0]:.1f}, end={results['soc'][-1]:.1f} MWh")

## 1. Asset Parameters

All physical and economic parameters of the system. These define the feasible region and cost structure of the MILP.

In [ ]:
# Display parameters as a readable table
param_groups = {
    "Time Resolution": {
        "Intervals (T)": f"{PARAMS['T']} (= 24h at 15-min)",
        "Interval duration (dt)": f"{PARAMS['dt']} h",
    },
    "Gas Grid": {
        "Gas price": f"{PARAMS['gas_price']} €/MWh_Hs",
        "CO2 factor": f"{PARAMS['co2_factor']} tCO2/MWh_Hs",
        "CO2 price": f"{PARAMS['co2_price']} €/tCO2",
        "Effective gas cost": f"{PARAMS['gas_cost']:.2f} €/MWh_Hs (gas + CO2)",
    },
    "Heat Pump": {
        "Electrical power": f"{PARAMS['hp_p_el_min']}–{PARAMS['hp_p_el_max']} MW_el",
        "COP": f"{PARAMS['hp_cop']} MW_th/MW_el",
        "Thermal output range": f"{PARAMS['hp_cop']*PARAMS['hp_p_el_min']:.1f}–{PARAMS['hp_cop']*PARAMS['hp_p_el_max']:.1f} MW_th",
    },
    "Condensing Boiler": {
        "Thermal power": f"{PARAMS['boiler_q_min']}–{PARAMS['boiler_q_max']} MW_th",
        "Efficiency": f"{PARAMS['boiler_eff']} MW_th/MW_Hs",
        "Min up/downtime": f"{PARAMS['boiler_min_up']*PARAMS['dt']:.0f}h / {PARAMS['boiler_min_down']*PARAMS['dt']:.0f}h",
    },
    "CHP": {
        "Electrical power": f"{PARAMS['chp_p_el_min']}–{PARAMS['chp_p_el_max']} MW_el",
        "Thermal output": f"{PARAMS['chp_heat_power_ratio']*PARAMS['chp_p_el_min']:.1f}–{PARAMS['chp_heat_power_ratio']*PARAMS['chp_p_el_max']:.1f} MW_th",
        "Electrical eff.": f"{PARAMS['chp_eff_el']}",
        "Thermal eff.": f"{PARAMS['chp_eff_th']}",
        "Min up/downtime": f"{PARAMS['chp_min_up']*PARAMS['dt']:.0f}h / {PARAMS['chp_min_down']*PARAMS['dt']:.0f}h",
        "Startup cost": f"{PARAMS['chp_startup_cost']:.0f} €",
    },
    "Thermal Storage": {
        "Capacity": f"{PARAMS['sto_capacity']} MWh_th",
        "Max charge/discharge": f"{PARAMS['sto_charge_max']} MW_th",
        "Loss per interval": f"{PARAMS['sto_loss']} MWh_th",
        "Initial SoC": f"{PARAMS['sto_soc_init']} MWh_th",
    },
}

for group, params in param_groups.items():
    print(f"\n{'─'*3} {group} {'─'*(50-len(group))}")
    for k, v in params.items():
        print(f"  {k:<30s} {v}")

## 2. Load Input Data

We need two time series for a given day:
- **Heat demand** (24 hourly values in MW_th)
- **Day-ahead electricity price** (24 hourly values in €/MWh)

The data loader reads from CSV files and handles timezone conversion (Berlin → UTC).

In [ ]:
# ── Data loading ──────────────────────────────────────────────────
# Point these paths to your local CSV files.
# Not committed to this repo.

DATA_DIR = Path("<path-to-data-directory>")

DEMAND_FILE = DATA_DIR / "<demand-csv-filename>.csv"
PRICE_FILE = DATA_DIR / "<price-csv-filename>.csv"

def load_heat_demand() -> pd.DataFrame:
    """Load full heat demand dataset. Returns DataFrame with UTC timestamps and MW_th."""
    df = pd.read_csv(DEMAND_FILE)
    df["timestamp"] = pd.to_datetime(df["Time Point"], utc=True)
    df["MW_th"] = df["Measured Heat Demand[W]"] / 1e6
    return df[["timestamp", "MW_th"]].set_index("timestamp")

def load_electricity_prices() -> pd.DataFrame:
    """Load DA electricity prices, deduplicated to hourly. Returns €/MWh."""
    df = pd.read_csv(PRICE_FILE, sep=";", decimal=",", low_memory=False)
    df["timestamp"] = pd.to_datetime(df["Datum von"], format="%d.%m.%Y %H:%M")
    df["timestamp"] = df["timestamp"].dt.tz_localize("Europe/Berlin", ambiguous="infer")
    df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")
    df["price_eur_mwh"] = df["Deutschland/Luxemburg [€/MWh] Originalauflösungen"]
    df = df.set_index("timestamp").resample("1h").first()
    return df[["price_eur_mwh"]]

def get_clean_days() -> list:
    """Return dates that have 24 complete, non-negative demand hours."""
    demand = load_heat_demand()
    demand["date"] = demand.index.date
    daily = demand.groupby("date")["MW_th"].agg(
        valid_count=lambda x: x.notna().sum(),
        has_negative=lambda x: (x < 0).any(),
    )
    clean = daily[(daily["valid_count"] == 24) & (~daily["has_negative"])]
    return sorted(clean.index.tolist())

def get_day_data(date) -> dict:
    """Get optimizer input for a single day: 24h demand + 24h prices."""
    if isinstance(date, str):
        date = pd.Timestamp(date).date()
    demand = load_heat_demand()
    prices = load_electricity_prices()
    day_start = pd.Timestamp(date, tz="UTC")
    day_end = day_start + pd.Timedelta("24h")
    d = demand.loc[day_start:day_end - pd.Timedelta("1h")]
    p = prices.loc[day_start:day_end - pd.Timedelta("1h")]
    if len(d) != 24:
        raise ValueError(f"Demand data incomplete for {date}: got {len(d)} hours, need 24")
    if len(p) != 24:
        raise ValueError(f"Price data incomplete for {date}: got {len(p)} hours, need 24")
    return {"demand_mw": d["MW_th"].tolist(), "price_eur": p["price_eur_mwh"].tolist(), "date": date}

print(f"Data directory: {DATA_DIR}")
print(f"Demand file exists: {DEMAND_FILE.exists()}")
print(f"Price file exists:  {PRICE_FILE.exists()}")

In [ ]:
# Pick a sample day and inspect the inputs
clean_days = get_clean_days()
print(f"Total clean days available: {len(clean_days)}")
print(f"Date range: {clean_days[0]} to {clean_days[-1]}")

# Use a winter day (high demand) for a more interesting dispatch
sample_date = clean_days[len(clean_days) // 4]  # ~25th percentile → winter/spring
data = get_day_data(sample_date)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.bar(range(24), data["demand_mw"], color="#e74c3c", alpha=0.7)
ax1.set_xlabel("Hour of day")
ax1.set_ylabel("MW_th")
ax1.set_title(f"Heat Demand — {sample_date}")
ax1.grid(True, alpha=0.3)

ax2.step(range(24), data["price_eur"], where="mid", color="#3498db", linewidth=1.5)
ax2.axhline(y=0, color="black", linewidth=0.5)
ax2.set_xlabel("Hour of day")
ax2.set_ylabel("€/MWh")
ax2.set_title(f"DA Electricity Price — {sample_date}")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDemand: min={min(data['demand_mw']):.1f}, max={max(data['demand_mw']):.1f}, "
      f"total={sum(data['demand_mw']):.0f} MWh_th")
print(f"Price:  min={min(data['price_eur']):.1f}, max={max(data['price_eur']):.1f}, "
      f"avg={np.mean(data['price_eur']):.1f} €/MWh")

## 3. Rolling Horizon Dispatch

Solve day-by-day across all clean days, carrying over the final state of each day
(SoC, unit on/off, time-in-state) as the initial condition for the next day.
No terminal SoC constraint — the storage level evolves naturally over time.

**Known limitation:** The optimizer is myopic — it only sees 24h ahead and tends to
drain storage toward end-of-day since there is no future cost signal. This leads to
suboptimal cross-day storage usage. One possible mitigation could be extending the
horizon (e.g. to 48h) using forecasted prices and demand, committing only the first
24h of decisions — but other approaches (terminal SoC penalties, value function
approximation, etc.) may also be viable. To be explored in future iterations.

In [ ]:
%%time
# ── Rolling horizon: solve all clean days sequentially ──────────────

import sys

all_results = []
carry_over = None  # first day uses defaults (all OFF, SoC=200)

for i, day in enumerate(clean_days):
    sys.stdout.write(f"\r  Day {i+1}/{len(clean_days)}: {day}")
    sys.stdout.flush()

    try:
        data = get_day_data(day)
    except ValueError:
        continue

    model = build_model(data["demand_mw"], data["price_eur"], initial_states=carry_over)
    res = solve(model)

    if not res["feasible"]:
        print(f"\n  WARNING: {day} infeasible ({res['status']}), resetting state")
        carry_over = None
        continue

    res["date"] = day
    all_results.append(res)
    carry_over = res["final_states"]

print(f"\n\nSolved {len(all_results)} / {len(clean_days)} days successfully")
print(f"Final SoC: {carry_over['soc_0']:.1f} MWh_th")

# Show summary for one example day
example_idx = len(all_results) // 4  # pick a winter day
print(f"\n{'='*60}")
print(f"Example day: {all_results[example_idx]['date']}")
print(f"{'='*60}")
print_summary(all_results[example_idx])

## 4. Visualize Results

Two views:
1. **Single-day dispatch** — detailed thermal dispatch, SoC, and price overlay for the example day
2. **Multi-day overview** — daily costs and SoC evolution across the full rolling horizon

In [ ]:
# ── Single-day dispatch plot (example day) ──────────────────────────
res = all_results[example_idx]
hours = np.array(res["hours"])

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1.5, 1.5]})

ax = axes[0]
hp = np.array(res["hp_q_th"])
boiler = np.array(res["boiler_q_th"])
chp = np.array(res["chp_q_th"])
discharge = np.array(res["sto_discharge"])
charge = np.array(res["sto_charge"])
demand = np.array(res["demand"])

ax.fill_between(hours, 0, hp, alpha=0.7, label="Heat Pump", color="#2ecc71", step="post")
ax.fill_between(hours, hp, hp + boiler, alpha=0.7, label="Boiler", color="#e67e22", step="post")
ax.fill_between(hours, hp + boiler, hp + boiler + chp, alpha=0.7, label="CHP", color="#9b59b6", step="post")
ax.fill_between(hours, hp + boiler + chp, hp + boiler + chp + discharge,
                alpha=0.5, label="Storage discharge", color="#3498db", step="post")
ax.fill_between(hours, 0, -charge, alpha=0.5, label="Storage charge", color="#3498db",
                hatch="//", step="post")
ax.step(hours, demand, where="post", color="red", linewidth=2, linestyle="--", label="Demand")
ax.set_ylabel("MW_th")
ax.set_title(f"Optimal Thermal Dispatch — {res['date']}")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 24)

ax = axes[1]
soc_hours = np.arange(0, len(res["soc"])) * PARAMS["dt"]
ax.fill_between(soc_hours, res["soc"], alpha=0.4, color="#3498db")
ax.plot(soc_hours, res["soc"], color="#2980b9", linewidth=1.5)
ax.axhline(y=PARAMS["sto_capacity"], color="gray", linestyle=":", alpha=0.5, label="Capacity")
ax.set_ylabel("SoC (MWh_th)")
ax.set_title("Thermal Storage — State of Charge")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[2]
da_price = np.array(res["da_price"])
ax.step(hours, da_price, where="post", color="#e67e22", linewidth=1.2)
ax.fill_between(hours, da_price, alpha=0.15, color="#e67e22", step="post")
hp_on = np.array(res["hp_on"], dtype=float)
chp_on = np.array(res["chp_on"], dtype=float)
ax.fill_between(hours, 0, hp_on * da_price.max() * 0.1, alpha=0.3, color="#2ecc71",
                label="HP on", step="post")
ax.fill_between(hours, 0, -chp_on * da_price.max() * 0.1, alpha=0.3, color="#9b59b6",
                label="CHP on", step="post")
ax.set_ylabel("€/MWh")
ax.set_xlabel("Hour of day")
ax.set_title("DA Price + Unit Activity")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Multi-day overview ───────────────────────────────────────────────
import datetime

dates = [r["date"] for r in all_results]
daily_costs = [r["cost_total"] for r in all_results]
end_socs = [r["soc"][-1] for r in all_results]
start_socs = [r["soc"][0] for r in all_results]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Daily cost
ax = axes[0]
colors = ["#2ecc71" if c < 0 else "#e74c3c" for c in daily_costs]
ax.bar(range(len(dates)), daily_costs, color=colors, alpha=0.7, width=1.0)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_ylabel("Daily net cost (€)")
ax.set_title("Rolling Horizon — Daily Costs (green = net revenue)")
ax.grid(True, alpha=0.3)

# SoC evolution
ax = axes[1]
ax.plot(range(len(dates)), start_socs, color="#2980b9", linewidth=1, alpha=0.5, label="Start SoC")
ax.plot(range(len(dates)), end_socs, color="#e74c3c", linewidth=1, label="End SoC")
ax.axhline(y=PARAMS["sto_capacity"], color="gray", linestyle=":", alpha=0.5)
ax.axhline(y=0, color="gray", linestyle=":", alpha=0.5)
ax.set_ylabel("SoC (MWh_th)")
ax.set_title("Storage State of Charge — Day Start vs End")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Daily demand
daily_demand = [sum(r["demand"]) * PARAMS["dt"] for r in all_results]
ax = axes[2]
ax.bar(range(len(dates)), daily_demand, color="#e74c3c", alpha=0.5, width=1.0)
ax.set_ylabel("MWh_th")
ax.set_xlabel("Day index")
ax.set_title("Daily Heat Demand")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary stats
print(f"Total days: {len(all_results)}")
print(f"Total cost: {sum(daily_costs):,.0f} €")
print(f"Avg daily cost: {np.mean(daily_costs):,.0f} €")
print(f"SoC range: {min(end_socs):.1f} – {max(end_socs):.1f} MWh_th")